# 01 · Limpieza de las imputaciones del MES

En las imputaciones está el detalle real de ejecución: qué máquina, qué tiempos reales, quién y cuándo.
Este notebook elimina los atributos que no se usan y limita las imputaciones al intervalo de estudio.

- **Entrada:** `data/raw/imputaciones.csv`
- **Salida:** `data/processed/df_imputations_limpieza.parquet` y `results/tables/embudo_01.json`

In [7]:
import sys
from pathlib import Path

# Raíz del repositorio (encontrar la carpeta que contiene src/) para poder importar src
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ)) # ubicar la ruta dentro del sys.path

import pandas as pd

from src.columnas import COLUMNAS_IMPUTACIONES, leer_csv
from src.embudo import guardar_embudo, tabla_embudo
from src.limpieza import eliminar_columnas, filtrar_por_fecha
from src.rutas import PROCESSED

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

embudo = {}  # json que usare para el número de imputaciones que quedan tras cada paso

## 1. Carga

In [8]:
df_imputations = leer_csv("imputaciones.csv", COLUMNAS_IMPUTACIONES)
embudo["Imputaciones iniciales"] = len(df_imputations)

print(df_imputations.shape)
df_imputations.head(3)

(196576, 26)


,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseId,smFaseNme,CantidadManualOKNbr,CantidadManualNOKNbr,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,PorcentajeDedicacionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,smFasePersonalizado1Txt,smNodoId2,smNodoPersonalizado1Nbr,smNodoPersonalizado2Nbr,ssEnumFaseEstadoId,SmFaseUid,TraspasadoInd,JsonBodyTxt,JsonResponseTxt
0,2026-08-05 00:57:09.823,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,OF2606156,PE1JNCX0012,Portacápsulas F704501049704(150713),30,TPP,0.0,0.0,0.00000,0.00000,0.0,0.0,1,NaN,NaN,NaN,TN09,NaN,NaN,INICIADA_OLANET,C0DF4C4C-2E19-483E-B9CD-6506FC7BABA0,1,"{""CodCompany"":""100"",""CodSite"":""01"",""Imputation...","{""Result"":true,""errorDetails"":[],""newIDImputat..."
1,2026-08-05 00:41:20.660,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,OF2606158,PE1XNCX0150,Elektrodenaufnahme F704501056335,35,TPP,3.0,NaN,40.46667,40.46667,0.0,100.0,1,NaN,NaN,NaN,TN09,NaN,NaN,FINALIZADA_OLANET,81CAF1D8-779C-45C7-B4D8-BF5FE5FE5969,1,"{""CodCompany"":""100"",""CodSite"":""01"",""Imputation...","{""Result"":true,""errorDetails"":[],""newIDImputat..."
2,2026-08-05 00:41:12.257,CM07,CH MAK. a61,OF2606232,BR2XA3X0275,COPPER PART Arm me095684s,35,CHO-PUN,0.0,0.0,0.00000,0.00000,0.0,0.0,1,NaN,NaN,NaN,CM07,NaN,NaN,INICIADA_OLANET,24A90559-B794-4C46-90E6-72B7A9A9FC79,1,"{""CodCompany"":""100"",""CodSite"":""01"",""Imputation...","{""Result"":true,""errorDetails"":[],""newIDImputat..."


## 2. Limpieza de atributos

Se eliminan del dataset los atributos prescindibles para el estudio.

In [9]:
COLUMNAS_SOBRAN = [
    "JsonResponseTxt", "JsonBodyTxt", "TraspasadoInd", "ssEnumFaseEstadoId",
    "smNodoPersonalizado2Nbr", "smNodoPersonalizado1Nbr", "smNodoId2",
    "smFasePersonalizado1Txt", "CantidadManualOKNbr", "CantidadManualNOKNbr", "smFaseId",
]

df_imputations = eliminar_columnas(df_imputations, COLUMNAS_SOBRAN)
df_imputations.head(3)

Eliminadas 11 columnas. Quedan 15 columnas y 196576 filas.


,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseNme,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,PorcentajeDedicacionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,SmFaseUid
0,2026-08-05 00:57:09.823,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,OF2606156,PE1JNCX0012,Portacápsulas F704501049704(150713),TPP,0.00000,0.00000,0.0,0.0,1,NaN,NaN,C0DF4C4C-2E19-483E-B9CD-6506FC7BABA0
1,2026-08-05 00:41:20.660,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,OF2606158,PE1XNCX0150,Elektrodenaufnahme F704501056335,TPP,40.46667,40.46667,0.0,100.0,1,NaN,NaN,81CAF1D8-779C-45C7-B4D8-BF5FE5FE5969
2,2026-08-05 00:41:12.257,CM07,CH MAK. a61,OF2606232,BR2XA3X0275,COPPER PART Arm me095684s,CHO-PUN,0.00000,0.00000,0.0,0.0,1,NaN,NaN,24A90559-B794-4C46-90E6-72B7A9A9FC79


## 3. Intervalo de estudio

Se trabaja con las imputaciones creadas entre el 1 de junio de 2025 y el 1 de agosto de 2026.
Son los mismos límites que se aplican a las órdenes de fabricación en el notebook 04:
antes de junio de 2025 el sistema estaba en fase de implantación y sus datos no son fiables,
y a partir de agosto no se puede garantizar que las órdenes estén terminadas.

In [10]:
df_imputations = filtrar_por_fecha(df_imputations, "FechaCreacionDte",
                                   desde="2025-06-01", hasta="2026-08-01",
                                   unidad="imputaciones")
embudo["Imputaciones dentro del intervalo temporal"] = len(df_imputations)

Partida: 196576 imputaciones
Desde 2025-06-01: 141329 (descartadas 55247)
Hasta 2026-08-01: 140485 (descartadas 844)


## 4. Guardado

In [11]:
df_imputations.to_parquet(PROCESSED / "df_imputations_limpieza.parquet")
guardar_embudo(embudo, "01")

## Resumen

In [12]:
tabla_embudo(embudo, "Número de imputaciones")

,Número de imputaciones
Imputaciones iniciales,196576
Imputaciones dentro del intervalo temporal,140485
